# Creative Test Results - Batch Evaluation

This notebook evaluates a batch of creative A/B test campaigns, calculates statistical results, uploads formatted outputs to BigQuery, and prepares an HTML summary email.

## Libraries

In [ ]:
import pandas as pd

from modules import production_ab_test_batch, batch_results_email

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

## Evaluation Configuration

Set the batch number, reporting window, and placeholder BigQuery table references.

In [ ]:
BATCH_NUMBER = 000
REPORTING_START_DATE = "YYYY-MM-DD"
REPORTING_END_DATE = "YYYY-MM-DD"

SCHEDULE_TABLE = "project.dataset.schedule_table"
RESULTS_TABLE = "project.dataset.results_table"

## Run Batch Results Workflow

The batch workflow pulls the schedule and performance data, runs each campaign through the A/B testing engine, and separates successful results from campaigns that need manual review.

In [ ]:
batch = production_ab_test_batch.ProductionABTestBatch(
    BATCH_NUMBER,
    REPORTING_START_DATE,
    REPORTING_END_DATE
)

schedule = batch.get_schedule(
    schedule_table=SCHEDULE_TABLE
)

performance = batch.get_performance()

get_results_response = batch.get_results(
    schedule,
    performance
)

results = get_results_response["results"]
issues = get_results_response["issues"]

results.head()

## Review Issues

Campaigns that fail during processing are captured separately so they can be reviewed without blocking the whole batch.

In [ ]:
issues

## Upload Results to BigQuery

The upload step checks whether results already exist before writing the batch output to the destination table.

In [ ]:
batch.upload_results_to_bq(
    results,
    RESULTS_TABLE
)

## Summary Email

The workflow can generate an HTML summary email containing result counts and a formatted table of variants compared with their controls.

In [ ]:
RECIPIENTS = [
    "recipient1@example.com",
    "recipient2@example.com"
]

In [ ]:
email = batch_results_email.BatchResultsEmail(
    batch_number=BATCH_NUMBER,
    email_to=RECIPIENTS
)

email_message = email.create_email(
    schedule_table=SCHEDULE_TABLE,
    results_table=RESULTS_TABLE
)

email_message

### Send Email

The send step is optional and can be skipped when testing the workflow locally.

In [ ]:
# Uncomment when ready to send
# email.send_email(email_message)